[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_10_reinforce_CartPole_complete.ipynb)

<div style="text-align:center">
    <h1>
        REINFORCE
    </h1>
</div>

<br><br>

<div style="text-align:center">
In this notebook we are going to implement the Monte Carlo version of Policy Gradient methods. The REINFORCE algorithm uses the full return to update the policy:
</div>

\begin{equation}
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+1} + \cdots + \gamma^{T-t-1} R_{T}
\end{equation}


<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_policy_network, plot_stats, seed_everything, plot_action_probs


## Import the necessary software libraries:

In [ ]:
# Standard tools for this notebook.
import os                       # used to count CPU cores for the parallel environments
import torch                    # PyTorch: builds the policy network and does tensor math
import gym                      # OpenAI Gym, provides the CartPole environment
import matplotlib.pyplot as plt # for drawing the environment and result plots
from tqdm import tqdm           # progress bar shown while training
from torch import nn as nn      # neural-network building blocks (layers, activations)
from torch.optim import AdamW   # optimiser that updates the policy weights

## Create and preprocess the environment

### Create the environment

In [ ]:
# Create the CartPole environment: balance a pole on a moving cart.
env = gym.make('CartPole-v0')

In [ ]:
# Number of values describing a state (4 for CartPole).
dims = env.observation_space.shape[0]
# Number of available actions (2: left or right).
actions = env.action_space.n

print(f"State dimensions: {dims}. Actions: {actions}")
print(f"Sample state: {env.reset()}")

In [ ]:
# Render one frame of the environment as an image.
plt.imshow(env.render(mode='rgb_array'))

### Prepare the environment to work with PyTorch

In [ ]:
# Wrapper that turns Gym's NumPy arrays into PyTorch tensors so the policy network can use them.
class PreprocessEnv(gym.Wrapper):

    def __init__(self, env):
        # Inherit all behaviour from the wrapped environment.
        gym.Wrapper.__init__(self, env)

    def reset(self):
        # Reset and convert the starting state to a float tensor.
        state = self.env.reset()
        return torch.from_numpy(state).float()

    def step(self, actions):
        # Convert the action tensor back to NumPy for Gym.
        actions = actions.squeeze().numpy()
        # Take the action(s) in the environment.
        next_state, reward, done, info = self.env.step(actions)
        # Convert everything back to tensors with the shapes the training loop expects.
        next_state = torch.from_numpy(next_state).float()
        reward = torch.tensor(reward).unsqueeze(1).float()
        done = torch.tensor(done).unsqueeze(1)
        return next_state, reward, done, info

In [ ]:
# Run several CartPole environments side by side to gather experience faster.
num_envs = os.cpu_count()
# A vectorised environment that steps all copies at once.
parallel_env = gym.vector.make('CartPole-v1', num_envs=num_envs)
# Seed everything for reproducibility.
seed_everything(parallel_env)
# Wrap it so it returns PyTorch tensors.
parallel_env = PreprocessEnv(parallel_env)

In [ ]:
# Reset all parallel environments to their starting states.
parallel_env.reset()

### Create the policy $\pi(s)$

In policy-gradient methods we don't learn action *values* and then act greedily. Instead we learn the **policy directly**: a network that outputs a probability for each action. The final `Softmax` layer guarantees the outputs are positive and sum to 1, so they form a proper probability distribution. At each step we then *sample* an action from this distribution.

In [ ]:
# The policy network: maps a state to a probability for each action.
# Unlike a Q-network it outputs *probabilities* (via Softmax), so we sample actions from it.
policy = nn.Sequential(
    nn.Linear(dims, 128),     # input layer: state (4 numbers) -> 128 hidden units
    nn.ReLU(),                # non-linear activation
    nn.Linear(128, 64),       # hidden layer: 128 -> 64
    nn.ReLU(),                # non-linear activation
    nn.Linear(64, actions),   # output layer: 64 -> one score per action
    nn.Softmax(dim=-1))       # turn the scores into probabilities that sum to 1

### Plot action probabilities

In [ ]:
# Three hand-made states to inspect what the policy 'thinks' before/after training.
neutral_state = torch.zeros(4)                  # cart centred, nothing moving
left_danger = torch.tensor([-2.3, 0., 0., 0.])  # cart far to the left
right_danger = torch.tensor([2.3, 0., 0., 0.])  # cart far to the right

#### Plot a neutral environment

In [ ]:
# Ask the policy for action probabilities in the neutral state and plot them.
probs = policy(neutral_state).detach().numpy()
plot_action_probs(probs=probs, labels=['Move Left', 'Move Right'])

#### Plot a state where the cart is too far left

In [ ]:
# Action probabilities when the cart is dangerously far left.
probs = policy(left_danger).detach().numpy()
plot_action_probs(probs=probs, labels=['Move Left', 'Move Right'])

#### Plot a state where the cart is too far right

In [ ]:
# Action probabilities when the cart is dangerously far right.
probs = policy(right_danger).detach().numpy()
plot_action_probs(probs=probs, labels=['Left', 'Right'])

## Implement the algorithm

### REINFORCE in plain words

REINFORCE is the simplest policy-gradient method. The recipe is:

1. **Play whole episodes** by sampling actions from the current policy (a *trajectory*).
2. For each step, compute the **return** $G_t$ -- the total discounted reward earned *from that step to the end of the episode*.
3. **Adjust the policy** so that actions which were followed by a high return become *more* likely, and actions followed by a low return become *less* likely.

The mathematical tool that makes step 3 possible is the **log-probability trick**. We can't back-propagate through the random "sample an action" step, but we *can* nudge the probability of the action we happened to take. The update for one step is

$$\nabla_\theta \, \log \pi(a_t \mid s_t)\; G_t,$$

i.e. the gradient of the log-probability of the chosen action, weighted by how good that episode turned out ($G_t$). Good returns push the log-prob up; bad returns push it down. We also add a small **entropy bonus** so the policy keeps exploring instead of collapsing onto one action too early.

Because we need the *full* return $G_t$, we wait until the episode finishes, then walk **backwards** through it accumulating $G = r_t + \gamma G$.

In [ ]:
# The REINFORCE algorithm: a Monte-Carlo policy-gradient method.
def reinforce(policy, episodes, alpha=1e-4, gamma=0.99):
    # Optimiser that updates the policy network.
    optim = AdamW(policy.parameters(), lr=alpha)
    # Track the policy-gradient loss and the return of each episode.
    stats = {'PG Loss': [], 'Returns': []}

    for episode in tqdm(range(1, episodes + 1)):
        # Reset all parallel environments.
        state = parallel_env.reset()
        # Track which environments have finished their episode.
        done_b = torch.zeros((num_envs, 1), dtype=torch.bool)
        # Store every (state, action, reward) we see; we learn only AFTER the episode ends.
        transitions = []
        ep_return = torch.zeros((num_envs, 1))

        # ----- Step 1: play out full trajectories by sampling from the policy -----
        while not done_b.all():
            # Sample an action from the policy's probability distribution.
            action = policy(state).multinomial(1).detach()
            # Take the action.
            next_state, reward, done, _ = parallel_env.step(action)
            # Save the transition (reward zeroed for envs that already finished).
            transitions.append([state, action, ~done_b * reward])
            ep_return += reward
            # Mark environments that just finished.
            done_b |= done
            state = next_state

        # ----- Step 2: walk backwards through the episode to compute returns and update -----
        # G will accumulate the discounted return from the end of the episode backwards.
        G = torch.zeros((num_envs, 1))
        for t, (state_t, action_t, reward_t) in reversed(list(enumerate(transitions))):
            # Return from time t: this reward plus the discounted return of the future.
            G = reward_t + gamma * G
            # Re-compute the action probabilities for this state (this time WITH gradients).
            probs_t = policy(state_t)
            # Log-probabilities (a tiny constant avoids log(0)).
            log_probs_t = torch.log(probs_t + 1e-6)
            # The log-probability of the action we actually took.
            action_log_prob_t = log_probs_t.gather(1, action_t)

            # Entropy of the policy: an exploration bonus that discourages it from becoming over-confident.
            entropy_t = - torch.sum(probs_t * log_probs_t, dim=-1, keepdim=True)
            # Discount factor for this time step.
            gamma_t = gamma ** t
            # Policy-gradient loss: push up the log-prob of good actions (high G), down for bad ones.
            pg_loss_t = - gamma_t * action_log_prob_t * G
            # Combine the policy loss with the entropy bonus and average over the parallel envs.
            total_loss_t = (pg_loss_t - 0.01 * entropy_t).mean()

            # Standard PyTorch update: zero grads, back-propagate, step.
            policy.zero_grad()
            total_loss_t.backward()
            optim.step()

        # Record stats for this episode.
        stats['PG Loss'].append(total_loss_t.item())
        stats['Returns'].append(ep_return.mean().item())

    return stats

In [ ]:
# Train the policy for 200 episodes.
stats = reinforce(policy, 200)

## Show results

### Show execution stats

In [ ]:
# Plot the policy-gradient loss and the episode returns.
plot_stats(stats)

### Plot action probabilities

#### Plot a neutral environment

In [ ]:
# After training: action probabilities in the neutral state.
probs = policy(neutral_state).detach().numpy()
plot_action_probs(probs=probs, labels=['Move Left', 'Move Right'])

#### Plot a state where the cart is too far left

In [ ]:
# After training: action probabilities when the cart is far left (should now strongly prefer moving right).
probs = policy(left_danger).detach().numpy()
plot_action_probs(probs=probs, labels=['Move Left', 'Move Right'])

#### Plot a state where the cart is too far right

In [ ]:
# After training: action probabilities when the cart is far right.
probs = policy(right_danger).detach().numpy()
plot_action_probs(probs=probs, labels=['Left', 'Right'])

### Test the resulting agent

In [ ]:
# Watch the trained policy play 5 episodes.
test_policy_network(env, policy, episodes=5)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch.13](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)